# Aktieanalys för AI-utvecklare

Projektet hämtar aktuella aktiekurser från Yahoo Finance API, jämför dem mot ett tänkt inköpspris, och klassificerar risk och utveckling per innehav.

In [2]:
# Importerar bibliotek: requests för API-anrop, csv och json för datahantering
import requests
import csv
import json

## Klasser: Aktie och Utdelningsaktie (arv)

In [3]:
# Basklass som representerar en vanlig aktie
class Aktie:
    def __init__(self, ticker, pris, antal):
        self.ticker = ticker
        self.pris = pris
        self.antal = antal

    def varde(self):
        # Räknar ut det totala värdet av innehavet
        return self.pris * self.antal

    def beskrivning(self):
        return f"{self.ticker}: {self.antal} st à {self.pris} kr = {self.varde():.2f} kr"

# Barnklass som ärver från Aktie och lägger till utdelning
class Utdelningsaktie(Aktie):
    def __init__(self, ticker, pris, antal, utdelning_per_aktie):
        super().__init__(ticker, pris, antal)
        self.utdelning_per_aktie = utdelning_per_aktie

    def arlig_utdelning(self):
        return self.utdelning_per_aktie * self.antal

    def direktavkastning(self):
        return self.utdelning_per_aktie / self.pris * 100

## Funktioner: hämta data, beräkna utveckling, klassificera risk

In [4]:
# Hämtar aktuellt pris för en given ticker via Yahoo Finance API
def hamta_pris(ticker):
    url = f"https://query2.finance.yahoo.com/v8/finance/chart/{ticker}"
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        svar = requests.get(url, headers=headers, timeout=5)
        data = svar.json()
        return data["chart"]["result"][0]["meta"]["regularMarketPrice"]
    except Exception as fel:
        # Fångar fel om API:et inte svarar eller returnerar oväntad data
        print(f"Kunde inte hämta pris för {ticker}: {fel}")
        return None

# Räknar ut procentuell värdeförändring sedan köptillfället
def berakna_vardeforandring(kop_pris, aktuellt_pris):
    forandring = aktuellt_pris - kop_pris
    return (forandring / kop_pris) * 100

# Klassificerar om utvecklingen är en uppgång, nedgång eller oförändrad
def bedom_utveckling(procent):
    if procent > 0:
        return "Uppgång"
    elif procent < 0:
        return "Nedgång"
    else:
        return "Oförändrad"

# Klassificerar risknivå baserat på hur stor förändringen är
def klassificera_risk(procent):
    if abs(procent) > 10:
        return "Hög risk"
    else:
        return "Låg risk"

## Bygg portföljen med riktig data från API

In [5]:
innehav = [
    {"ticker": "VOLV-B.ST", "antal": 10, "kop_pris": 250, "utdelning": 6.50},
    {"ticker": "ERIC-B.ST", "antal": 25, "kop_pris": 70, "utdelning": None},
    {"ticker": "HM-B.ST", "antal": 5, "kop_pris": 150, "utdelning": 9.75},
]

portfolj = []
for post in innehav:
    pris = hamta_pris(post["ticker"])
    if pris is None:
        continue
    if post["utdelning"] is not None:
        aktie = Utdelningsaktie(post["ticker"], pris, post["antal"], post["utdelning"])
    else:
        aktie = Aktie(post["ticker"], pris, post["antal"])
    portfolj.append({"objekt": aktie, "kop_pris": post["kop_pris"]})

portfoljvarde = 0
for post in portfolj:
    aktie = post["objekt"]
    procent = berakna_vardeforandring(post["kop_pris"], aktie.pris)
    utveckling = bedom_utveckling(procent)
    risk = klassificera_risk(procent)

    print(aktie.beskrivning())
    print(f"  Förändring sedan köp: {procent:.1f}% ({utveckling}, {risk})")
    if isinstance(aktie, Utdelningsaktie):
        print(f"  Direktavkastning: {aktie.direktavkastning():.2f}%")

    portfoljvarde += aktie.varde()

print(f"\nTotalt portföljvärde: {portfoljvarde:.2f} kr")

VOLV-B.ST: 10 st à 318.4 kr = 3184.00 kr
  Förändring sedan köp: 27.4% (Uppgång, Hög risk)
  Direktavkastning: 2.04%
ERIC-B.ST: 25 st à 91.56 kr = 2289.00 kr
  Förändring sedan köp: 30.8% (Uppgång, Hög risk)
HM-B.ST: 5 st à 158.35 kr = 791.75 kr
  Förändring sedan köp: 5.6% (Uppgång, Låg risk)
  Direktavkastning: 6.16%

Totalt portföljvärde: 6264.75 kr


## Spara resultat till fil

In [6]:
data_att_spara = []
for post in portfolj:
    aktie = post["objekt"]
    data_att_spara.append({
        "ticker": aktie.ticker,
        "pris": aktie.pris,
        "antal": aktie.antal,
        "varde": round(aktie.varde(), 2)
    })

with open("portfolj.json", "w") as fil:
    json.dump(data_att_spara, fil, indent=2)

print("Sparat till portfolj.json")

# Sparar portföljen som CSV också, utöver JSON
with open("portfolj.csv", "w", newline="") as csvfil:
    writer = csv.writer(csvfil)
    writer.writerow(["ticker", "pris", "antal", "varde"])
    for rad in data_att_spara:
        writer.writerow([rad["ticker"], rad["pris"], rad["antal"], rad["varde"]])

print("Sparat till portfolj.csv")

Sparat till portfolj.json
Sparat till portfolj.csv
